In [68]:
import numpy as np
import random
import sys

sys.path.append("../")
from Node import Node

In [69]:
def create_start():
    S = []
    for i in range(0, 2):
        arr = np.random.randint(0, 2, (4, 4))
        arr = arr.astype(str)
        
        x = random.randint(0, 3)
        y = random.randint(0, 3)
        
        if arr[x, y] == '1':
            arr[x, y] = 'd'
        else:
            arr[x, y] = 'x'
        S.append(arr)
        
    return S

In [70]:
def find_vaccuum_position(matrix):
    pos = np.argwhere((matrix == 'x') | (matrix == 'd'))
    if len(pos) == 0:
        raise ValueError("Vacuum cleaner not found in the matrix!")
    return pos[0][0], pos[0][1]

In [71]:
def moves(matrix):
    lst_move = []
    row, col = find_vaccuum_position(matrix)
    
    directions = [
        ['LEFT', 0, -1],
        ['RIGHT', 0, 1],
        ['UP', -1, 0],
        ['DOWN', 1, 0],
        ['SUCK', 0, 0] 
    ]
    
    for dir, x, y in directions:
        row_new = row + x
        col_new = col + y
        
        if row_new < 4 and row_new >= 0 and col_new < 4 and col_new >= 0:
            lst_move.append([dir, row_new, col_new])
            
    return lst_move

In [72]:
def solution(node):
    path = []
    
    while node.parent is not None:
        path.append((node.action, node.state))
        node = node.parent
    
    path.reverse()
    
    return path

In [73]:
def count_dirty(matrix):
    count = 0
    for x in range(0, 4):
        for y in range(0, 4):
            if matrix[x][y] == '1' or matrix[x][y] == 'd':
                count += 1
    return count

In [74]:
def check_goal(lst):
    count = 0
    for x in lst:
        if count_dirty(x) == 0:
            count += 1
            
    if count == len(lst):
        return True
    
    return False

In [75]:
def observe(current_state, action):

    row, col = find_vaccuum_position(current_state)
    for dir, x, y in moves(current_state):
        if dir == action:
            return current_state[x, y]
        
    return 'WALL'

In [76]:
def transition(state, action):
    child = state.copy()
    row, col = find_vaccuum_position(child)
    
    for dir, x, y in moves(child):
        if dir == action:
            child[row, col] = '0' 
            child[x, y] = 'x'
            break
    return child

In [77]:
def partially_observation(start, partial):
    print("TRANG THAI BAN DAU AN DUNG (BELIEF):\n", *start, sep="\n")
    print("TRANG THAI THUC (CO O AN '?'):\n", partial, "\n")

    root = Node(start, None, None, 0)
    root.partial = partial.copy()

    frontier, reached = [root], set()

    while frontier:
        node = frontier.pop()
        belief, current_real = node.state, node.partial

        key = tuple(tuple(map(tuple, s)) for s in belief)
        if key in reached: 
            continue
        reached.add(key)

        if check_goal(belief):
            print("TIM THAY LOI GIAI")
            solutions = solution(node)
            print(f"Huong di chuyen: {[x[0] for x in solutions]}")
            for x in solutions: print(x[1])
            return

        actions = {dir for state in belief for dir, _, _ in moves(state)}

        for action in actions:
            real_child = transition(current_real, action)
            observation = observe(real_child, action) 

            new_belief = []
            for state in belief:
                next_state = transition(state, action)
                if observe(next_state, action) == observation:
                    new_belief.append(next_state)

            if not new_belief: continue

            unique_belief = []
            seen_states = set()
            for s in new_belief:
                s_tuple = tuple(map(tuple, s))
                if s_tuple not in seen_states:
                    seen_states.add(s_tuple)
                    unique_belief.append(s)

            child_key = tuple(tuple(map(tuple, s)) for s in unique_belief)
            if child_key not in reached:
                child_node = Node(unique_belief, node, action, node.cost + 1)
                child_node.partial = real_child
                frontier.append(child_node)

    print("KHONG TIM THAY LOI GIAI")
    return None

In [78]:
if __name__ == "__main__":
    
    start = [np.array([['1', '0', '1', '0'],
                       ['0', '0', '0', '1'],
                       ['x', '1', '0', '1'],
                       ['0', '0', '0', '1']]),
             
             np.array([['1', '0', '0', '0'],
                       ['1', '1', '1', 'x'],
                       ['1', '0', '0', '0'],
                       ['1', '1', '0', '0']])]
    
    partial = np.array([['1', '?', '1', '0'],
                        ['0', '0', '?', '1'],
                        ['x', '1', '0', '1'],
                        ['?', '?', '?', '?']])
    
    partially_observation(start, partial)

TRANG THAI BAN DAU AN DUNG (BELIEF):

[['1' '0' '1' '0']
 ['0' '0' '0' '1']
 ['x' '1' '0' '1']
 ['0' '0' '0' '1']]
[['1' '0' '0' '0']
 ['1' '1' '1' 'x']
 ['1' '0' '0' '0']
 ['1' '1' '0' '0']]
TRANG THAI THUC (CO O AN '?'):
 [['1' '?' '1' '0']
 ['0' '0' '?' '1']
 ['x' '1' '0' '1']
 ['?' '?' '?' '?']] 

TIM THAY LOI GIAI
Huong di chuyen: ['LEFT', 'RIGHT', 'RIGHT', 'RIGHT', 'DOWN', 'UP', 'UP', 'DOWN', 'LEFT', 'UP', 'UP', 'DOWN', 'LEFT', 'UP', 'LEFT']
[array([['1', '0', '1', '0'],
       ['0', '0', '0', '1'],
       ['x', '1', '0', '1'],
       ['0', '0', '0', '1']], dtype='<U1')]
[array([['1', '0', '1', '0'],
       ['0', '0', '0', '1'],
       ['0', 'x', '0', '1'],
       ['0', '0', '0', '1']], dtype='<U1')]
[array([['1', '0', '1', '0'],
       ['0', '0', '0', '1'],
       ['0', '0', 'x', '1'],
       ['0', '0', '0', '1']], dtype='<U1')]
[array([['1', '0', '1', '0'],
       ['0', '0', '0', '1'],
       ['0', '0', '0', 'x'],
       ['0', '0', '0', '1']], dtype='<U1')]
[array([['1', '0', '